### Notebook Purpose: Feature Engineering Pipeline
- Document the leakage-safe feature pipeline reused by all modeling notebooks
- Build 15-minute per-ID grids and derive time/lag/rolling features
- Run leakage checks to ensure train-only statistics stay isolated
- Persist representative feature grids/artifacts to `results/` for sanity checks

### Note
- Shared inputs/outputs and execution conventions are documented in the project README.


In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Minimal environment setup to keep the notebook focused on the core experiments.
# If you run this elsewhere, update PROJECT_DIR accordingly.
PROJECT_DIR = Path("/content/drive/MyDrive/0.Portfolio/electricity_price_forecasting").resolve()


In [ ]:
project_str = str(PROJECT_DIR)
if project_str not in sys.path:
    sys.path.insert(0, project_str)

DATA_DIR         = PROJECT_DIR / "data"
RESULTS_DIR      = PROJECT_DIR / "results"
METRICS_DIR      = RESULTS_DIR / "metrics"
MODELS_DIR       = RESULTS_DIR / "models"
WINDOW_INDEX_DIR = RESULTS_DIR / "window_index"
LOGS_DIR         = RESULTS_DIR / "logs"
CACHE_DIR        = RESULTS_DIR / "cache"
PLOTS_DIR        = RESULTS_DIR / "plots"

for p in [
    RESULTS_DIR,
    PLOTS_DIR,
    METRICS_DIR,
    MODELS_DIR,
    WINDOW_INDEX_DIR,
    LOGS_DIR,
    CACHE_DIR,
    PLOTS_DIR / "models",
    PLOTS_DIR / "eval/curves",
    PLOTS_DIR / "eval/tables",
    PLOTS_DIR / "eda",
    PLOTS_DIR / "feature_engineering",
]:
    p.mkdir(parents=True, exist_ok=True)


# 02 — Feature Engineering (Leakage-Safe, Window-Compatible)

This notebook documents the **exact feature pipeline** used by the modeling notebooks (03–05) and explains:

- What features are created (and why)
- How we prevent leakage (split boundaries + lag/rolling definitions)
- Where a *fit-based scaler* would be fit (train only) and how to apply it
- How a **continuous per-ID block** is created (15‑minute grid with `pos`)
- A small **sanity check** that windows align with targets (WIN/HORIZON)

All code / comments / prints are in English.

In [3]:
# === Canonical constants (shared) ===

WIN = 128
HORIZON = 10
FREQ = "15min"

TRAIN_START = pd.Timestamp("2021-01-01 00:00:00", tz="UTC")
TRAIN_END   = pd.Timestamp("2023-12-31 23:45:00", tz="UTC")
TEST_START  = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
TEST_END    = pd.Timestamp("2024-12-31 23:45:00", tz="UTC")

TARGET_COLS = ["high", "low", "close", "volume"]
VOLUME_COL = "volume"

print("WIN:", WIN, "| HORIZON:", HORIZON, "| history=", WIN*15, "min", "| horizon=", HORIZON*15, "min")

WIN: 128 | HORIZON: 10 | history= 1920 min | horizon= 150 min


## 1) Continuous blocks: single-ID 15‑minute grid

A “continuous block” means:
- Each ID has rows at every 15‑minute step between fixed bounds
- A `pos` column enumerates 0..T-1
- This makes `(ID_code, start_pos)` a stable window coordinate

The grid builder ensures **no NaNs**:
- `volume`: missing → 0
- prices: ffill/bfill (and final fill 0 as a guard)

In [4]:
# === Stable ID mapping ===

from electricity_forecasting.io.id_mapping import (
    load_id_mapping_json,
    save_id_mapping_json,
    scan_unique_ids_from_parquet,
    build_id_mapping,
)

ID_MAP_PATH = RESULTS_DIR / "window_index" / "id_code_mapping.json"

if ID_MAP_PATH.exists():
    ID_TO_CODE, CODE_TO_ID, ALL_CODES = load_id_mapping_json(ID_MAP_PATH)
    print("OK: Loaded cached ID mapping | n_ids:", len(ALL_CODES))
else:
    ids_raw = scan_unique_ids_from_parquet(str(TEST_PARQUET), id_col="ID", batch_size=1_000_000)
    ID_TO_CODE, CODE_TO_ID, ALL_CODES = build_id_mapping(ids_raw)
    save_id_mapping_json(ID_MAP_PATH, id_to_code=ID_TO_CODE, code_to_id=CODE_TO_ID)
    print("OK: Scanned and saved ID mapping | n_ids:", len(ALL_CODES))

OK: Loaded cached ID mapping | n_ids: 672


In [5]:
# === Continuous grid for train ID ===

from electricity_forecasting.io.parquet_io import load_parquet_filtered
from electricity_forecasting.io.schema import enforce_schema
from electricity_forecasting.preprocessing.features import build_15min_grid

sample_code = random.choice(ALL_CODES)
raw_id = str(CODE_TO_ID[sample_code])
print("Sample ID_code:", sample_code, "| raw_id:", raw_id)

df_obs = load_parquet_filtered(
    parquet_path=str(TRAIN_PARQUET),
    ids=[raw_id],
    columns=["ExecutionTime", "ID", "high", "low", "close", "volume"],
    time_min=TRAIN_START,
    time_max=TRAIN_END,
)
df_obs = enforce_schema(df_obs)

df_grid = build_15min_grid(
    df_obs=df_obs,
    ids=None,
    start_utc=TRAIN_START,
    end_utc=TRAIN_END,
    freq=FREQ,
    id_col="ID",
    time_col="ExecutionTime",
    price_cols=("high", "low", "close"),
    volume_col="volume",
    add_pos=True,
)

print("Grid length T:", len(df_grid))
print("Any NaNs?", df_grid[TARGET_COLS].isna().any().to_dict())

t = pd.to_datetime(df_grid["ExecutionTime"], utc=True)
dt_ok = (t.diff().dropna() == pd.Timedelta("15min")).mean()
pos_ok = (df_grid["pos"].diff().dropna() == 1).mean()
print("Share of 15-min deltas:", float(dt_ok))
print("Share of pos increments:", float(pos_ok))

Sample ID_code: 654 | raw_id: Wed19Q3
Grid length T: 105120
Any NaNs? {'high': False, 'low': False, 'close': False, 'volume': False}
Share of 15-min deltas: 1.0
Share of pos increments: 1.0


## 2) Feature set used in modeling notebooks (03–05)

**Targets (supervised):**
- `high`, `low`, `close`, `volume` (10 steps ahead)

**Key transforms:**
- `logvol = log1p(volume)` for stability under heavy-tailed volumes

**Time features (Berlin time):**
- hour / day-of-week / month
- optional cyclical hour (sin/cos)

**Lag / rolling features (leakage-safe):**
- lags on `logvol`: 1, 2
- rolling mean on **shifted** `logvol` (shift=1): windows 4, 16

We keep the feature list ordering fixed using `infer_feature_cols(...)` so that all notebooks agree on column order.

In [6]:
# === Feature engineering pipeline ===

from IPython.display import display

df_feat = df_grid.copy()

# Pointwise transform (no leakage): reduces scale variance
df_feat["logvol"] = np.log1p(df_feat["volume"].astype(np.float32)).astype(np.float32)

# Berlin time features
df_feat = add_time_features(
    df_feat,
    time_col="ExecutionTime",
    tz="Europe/Berlin",
    prefix="berlin",
    add_cyclical=True,
)

# Leakage-safe lags/rolling: based on PAST values only
df_feat = add_lag_features(
    df_feat,
    cols=("logvol",),
    lags=(1, 2),
    id_col="ID",
    time_col="ExecutionTime",
)

df_feat = add_rolling_features(
    df_feat,
    cols=("logvol",),
    windows=(4, 16),
    shift=1,              # critical: uses history, not current timestep
    min_periods=1,
    id_col="ID",
    time_col="ExecutionTime",
)

FEATURE_COLS = infer_feature_cols(df_feat)
print("FEATURE_COLS (order fixed):")
for c in FEATURE_COLS:
    print(" -", c)

print("n_features:", len(FEATURE_COLS))
print("Example rows:")
display(df_feat[["ExecutionTime", "ID", "pos"] + FEATURE_COLS + TARGET_COLS].head(5))

FEATURE_COLS (order fixed):
 - logvol
 - logvol_lag1
 - logvol_lag2
 - logvol_rollmean_16
 - logvol_rollmean_4
 - close
 - high
 - low
 - dow_berlin
 - hour_berlin
 - hour_berlin_cos
 - hour_berlin_sin
 - month_berlin
n_features: 13
Example rows:


,ExecutionTime,ID,pos,logvol,logvol_lag1,logvol_lag2,logvol_rollmean_16,logvol_rollmean_4,close,high,low,dow_berlin,hour_berlin,hour_berlin_cos,hour_berlin_sin,month_berlin,high,low,close,volume
0,2021-01-01 00:00:00+00:00,Wed19Q3,0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,4,1,0.965926,0.258819,1,0.0,0.0,0.0,0.0
1,2021-01-01 00:15:00+00:00,Wed19Q3,1,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,4,1,0.965926,0.258819,1,0.0,0.0,0.0,0.0
2,2021-01-01 00:30:00+00:00,Wed19Q3,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,1,0.965926,0.258819,1,0.0,0.0,0.0,0.0
3,2021-01-01 00:45:00+00:00,Wed19Q3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,1,0.965926,0.258819,1,0.0,0.0,0.0,0.0
4,2021-01-01 01:00:00+00:00,Wed19Q3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,2,0.866025,0.500000,1,0.0,0.0,0.0,0.0


## 3) Leakage prevention (what is “safe” and why)

**Leakage to avoid:**  
Any feature at time *t* must not use information from:
- the target horizon (t+1 ... t+HORIZON), or
- validation/test periods during fitting.

We enforce safety in two layers:

1) **Split policy**
- Train: 2021–2023
- Validation: last 10% of **train time** (within 2021–2023)
- Test: 2024 (evaluated once at the end)

2) **Feature definitions**
- `logvol_lag_k(t) = logvol(t-k)`
- `logvol_rollmean_w(t) = mean(logvol(t-1), ..., logvol(t-w))` (shift=1)

So features at window end time `t_end` never peek at future steps in `Y`.

In [9]:
# === Leakage check (single row) ===

idx = int(1000)

row = df_feat.iloc[idx]
prev1 = df_feat.iloc[idx-1]["logvol"]
prev2 = df_feat.iloc[idx-2]["logvol"]

print("logvol(t)        :", float(row["logvol"]))
print("logvol_lag1(t)   :", float(row["logvol_lag1"]), "| should equal logvol(t-1) =", float(prev1))
print("logvol_lag2(t)   :", float(row["logvol_lag2"]), "| should equal logvol(t-2) =", float(prev2))

# Rolling mean uses shifted series (includes t-1, not t)
w = 4
manual = df_feat.iloc[idx-w:idx]["logvol"].mean()  # idx-w..idx-1
print("logvol_rollmean_4:", float(row["logvol_rollmean_4"]), "| manual:", float(manual))

logvol(t)        : 0.0
logvol_lag1(t)   : 0.0 | should equal logvol(t-1) = 0.0
logvol_lag2(t)   : 0.0 | should equal logvol(t-2) = 0.0
logvol_rollmean_4: 0.0 | manual: 0.0


## 4) Where to fit a scaler (train only)

Tree models (LGBM) do not require scaling, but DL models often benefit.

To keep the pipeline portable, we *optionally* standardize a chosen subset of numeric features using
**train period only (2021–2023)**.

Below is a minimal, dependency-free scaler that:
- fits on `train_mask` rows only
- then transforms *any* split (train/val/test) without leakage.

In [10]:
# === SimpleStandardScaler helper ===

class SimpleStandardScaler:
    def __init__(self, eps: float = 1e-8):
        self.eps = float(eps)
        self.mean_ = None
        self.std_ = None

    def fit(self, X: np.ndarray):
        X = np.asarray(X, dtype=np.float32)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_ = np.maximum(self.std_, self.eps)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float32)
        return (X - self.mean_) / self.std_

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)

# Example: fit scaler on train-only rows for a small set of numeric columns
NUMERIC_TO_SCALE = ["logvol", "close", "high", "low"]

train_mask = (
    (pd.to_datetime(df_feat["ExecutionTime"], utc=True) >= TRAIN_START)
    & (pd.to_datetime(df_feat["ExecutionTime"], utc=True) <= TRAIN_END)
)

X_train = df_feat.loc[train_mask, NUMERIC_TO_SCALE].to_numpy(np.float32)

scaler = SimpleStandardScaler().fit(X_train)
X_scaled = scaler.transform(df_feat[NUMERIC_TO_SCALE].to_numpy(np.float32))

print("Scaled feature means (all rows, should be ~0 if stationary):")
print(np.round(X_scaled.mean(axis=0), 3))
print("Scaled feature stds  (all rows, should be ~1):")
print(np.round(X_scaled.std(axis=0), 3))

# Note: In the modeling notebooks, we rely primarily on log1p(volume) and model normalization.
# This scaler block is provided to document leakage-safe fitting and to enable scaling if desired.

Scaled feature means (all rows, should be ~0 if stationary):
[-0. -0. -0.  0.]
Scaled feature stds  (all rows, should be ~1):
[1. 1. 1. 1.]


## 5) Small-sample sanity check: one window → targets

We verify the canonical window shapes:

- `X`: (WIN, F)
- `Y`: (HORIZON, C) where C=4 targets (high/low/close/volume)

And we verify the timestamp alignment:
- `X` ends at `t_end`
- `Y` starts at `t_end + 15min` (next step)

In [11]:
# === Build sample training window ===

T = len(df_feat)
max_start = T - WIN - HORIZON
assert max_start > 0, "Not enough data to build a window."

start_pos = int(1000)
assert start_pos <= max_start

g0 = start_pos
g1 = start_pos + WIN
g2 = g1 + HORIZON

X = df_feat.iloc[g0:g1][FEATURE_COLS].to_numpy(np.float32)
Y = df_feat.iloc[g1:g2][TARGET_COLS].to_numpy(np.float32)

print("X shape:", X.shape, " (WIN, F)")
print("Y shape:", Y.shape, " (HORIZON, C)")

t_end = pd.to_datetime(df_feat.iloc[g1-1]["ExecutionTime"], utc=True)
t_y0  = pd.to_datetime(df_feat.iloc[g1]["ExecutionTime"], utc=True)

print("X end time:", t_end)
print("Y start   :", t_y0)
print("Delta     :", t_y0 - t_end)

print("Any NaNs in X?", bool(np.isnan(X).any()))
print("Any NaNs in Y?", bool(np.isnan(Y).any()))

X shape: (128, 13)  (WIN, F)
Y shape: (10, 4)  (HORIZON, C)
X end time: 2021-01-12 17:45:00+00:00
Y start   : 2021-01-12 18:00:00+00:00
Delta     : 0 days 00:15:00
Any NaNs in X? False
Any NaNs in Y? False


## 6) Summary: features that feed notebooks 03–05

The modeling notebooks use the same canonical feature pipeline:
- Continuous 15‑minute grid per ID (`pos` indexing)
- `logvol` + lag/rolling features (leakage-safe)
- Berlin-time features for periodicity
- Window-based sampling defined by `(ID_code, start_pos)`